# Lab Assignment 1

In [1]:
!pip install annoy faiss-cpu hnswlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 21.8 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 184.0 MB/s  0:00:00
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=521282 sha256=69fa168ea521a1038d5b85522a53ff56175ef2087247de6284d8cdcb04830923
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2683507 sha256=2e087b66bb618066b7a50c98873b1027f406cc0b0f9e99fb6858c5c70cb007f7
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f

In [2]:
import time
import numpy as np
import pandas as pd
from annoy import AnnoyIndex
import faiss
import hnswlib

In [3]:
np.random.seed(42)

data_sizes = [1_000, 1_000_000]
dimensions = [2, 5]
metrics = ["Euclidean (L2)", "Cosine / Angular"]
k = 10

results = []

for metric in metrics:
    for n_data in data_sizes:
        for dim in dimensions:
            print(f"Running: {metric} | {n_data:,} points | {dim}D...")

            # Generate synthetic float32 dataset and query
            X = np.random.random((n_data, dim)).astype(np.float32)
            query = np.random.random((1, dim)).astype(np.float32)

            # Normalize vectors if testing Cosine/Angular similarity
            if metric == "Cosine / Angular":
                faiss.normalize_L2(X)
                faiss.normalize_L2(query)

            # Ground Truth

            if metric == "Euclidean (L2)":
                gt_index = faiss.IndexFlatL2(dim)
            else:
                gt_index = faiss.IndexFlatIP(dim)
            gt_index.add(X)
            _, gt_indices = gt_index.search(query, k)
            gt_set = set(gt_indices[0])

            # 1. ANNOY
            annoy_metric = 'euclidean' if metric == "Euclidean (L2)" else 'angular'
            ann_index = AnnoyIndex(dim, annoy_metric)

            t0 = time.perf_counter()
            for i in range(n_data):
                ann_index.add_item(i, X[i])
            ann_index.build(10)  # 10 trees
            ann_build = time.perf_counter() - t0

            t0 = time.perf_counter()
            ann_neighbors = ann_index.get_nns_by_vector(query[0], k)
            ann_query = (time.perf_counter() - t0) * 1000  # in milliseconds
            ann_recall = len(gt_set.intersection(set(ann_neighbors))) / k

            # 2. FAISS (IndexIVFFlat for ANN, or IndexFlat for small N)

            nlist = 100 if n_data >= 10_000 else 10
            if metric == "Euclidean (L2)":
                quantizer = faiss.IndexFlatL2(dim)
                faiss_index = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_L2)
            else:
                quantizer = faiss.IndexFlatIP(dim)
                faiss_index = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_INNER_PRODUCT)

            t0 = time.perf_counter()
            faiss_index.train(X)
            faiss_index.add(X)
            faiss_build = time.perf_counter() - t0

            faiss_index.nprobe = 5
            t0 = time.perf_counter()
            _, faiss_indices = faiss_index.search(query, k)
            faiss_query = (time.perf_counter() - t0) * 1000  # in milliseconds
            faiss_recall = len(gt_set.intersection(set(faiss_indices[0]))) / k

            # 3. HNSW (hnswlib)

            hnsw_space = 'l2' if metric == "Euclidean (L2)" else 'cosine'
            hnsw_index = hnswlib.Index(space=hnsw_space, dim=dim)

            t0 = time.perf_counter()
            hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
            hnsw_index.add_items(X)
            hnsw_build = time.perf_counter() - t0

            hnsw_index.set_ef(50)
            t0 = time.perf_counter()
            hnsw_labels, _ = hnsw_index.knn_query(query, k=k)
            hnsw_query = (time.perf_counter() - t0) * 1000  # in milliseconds
            hnsw_recall = len(gt_set.intersection(set(hnsw_labels[0]))) / k

            # Record row
            results.append({
                "Metric": metric,
                "Data / Dim": f"{n_data:,} / {dim}D",
                "ANNOY (Build | Query | Recall)": f"{ann_build:.4f}s | {ann_query:.3f}ms | {ann_recall:.1f}",
                "FAISS (Build | Query | Recall)": f"{faiss_build:.4f}s | {faiss_query:.3f}ms | {faiss_recall:.1f}",
                "HNSW (Build | Query | Recall)": f"{hnsw_build:.4f}s | {hnsw_query:.3f}ms | {hnsw_recall:.1f}",
            })

# Display formatted table
df_results = pd.DataFrame(results)
print("\n=== LAB ASSIGNMENT 1 BENCHMARK RESULTS ===")
print(df_results.to_markdown(index=False))

Running: Euclidean (L2) | 1,000 points | 2D...
Running: Euclidean (L2) | 1,000 points | 5D...
Running: Euclidean (L2) | 1,000,000 points | 2D...
Running: Euclidean (L2) | 1,000,000 points | 5D...
Running: Cosine / Angular | 1,000 points | 2D...
Running: Cosine / Angular | 1,000 points | 5D...
Running: Cosine / Angular | 1,000,000 points | 2D...
Running: Cosine / Angular | 1,000,000 points | 5D...

=== LAB ASSIGNMENT 1 BENCHMARK RESULTS ===
| Metric           | Data / Dim     | ANNOY (Build | Query | Recall)   | FAISS (Build | Query | Recall)   | HNSW (Build | Query | Recall)   |
|:-----------------|:---------------|:---------------------------------|:---------------------------------|:--------------------------------|
| Euclidean (L2)   | 1,000 / 2D     | 0.0065s | 0.046ms | 1.0          | 0.0011s | 0.047ms | 1.0          | 0.0146s | 0.052ms | 1.0         |
| Euclidean (L2)   | 1,000 / 5D     | 0.0048s | 0.038ms | 1.0          | 0.0010s | 0.044ms | 1.0          | 0.0151s | 0.070ms | 1.